# Распознавание аккордов в аудио: итоговая модель

**Магистерская диссертация** «Development of Machine Learning Models for Musical Chord Recognition in Digital Audio».
Нұрым Абзал, Astana IT University. Состояние на 20.11.2026.

Это чистая копия проекта: только код итогового рецепта (пакет `chordrec`), веса 15 обученных моделей и сохранённые числа.
Ноутбук показывает, где в коде каждый механизм и каждое правило распознавания, сколько каждый дал, итоговые результаты
и распознаёт аккорды в аудиозаписи прямо здесь. Код проверен против исследовательского кода, на котором получены числа
диссертации: те же веса дают те же ответы (`tests/check_against_research.py`, раздел 8).

## Как читать числа
- Мера — взвешенная полнота символов аккордов (WCSR, `mir_eval`): доля времени записи с верным ответом на данном уровне, %.
- Уровни: **root** — основной тон; **majmin** — мажор или минор (C, Cm); **triads** — трезвучие целиком (C, Cm, Cdim, Caug, Csus4);
  **sevenths** — мажор, минор и септаккорды C7, Cmaj7, Cm7; **tetrads** — все четырёхзвучия (Cdim7, Cm7b5 и др.);
  **MIREX** — совпадают хотя бы три звука аккорда.
- «15 моделей» — 5 фолдов кросс-валидации по группам GuitarSet × 3 seeds. Разница рецептов — среднее 15 пар;
  «фолдов > 0» — в скольких фолдах из 5 она положительна.
- Наборы: **GuitarSet DEV** — 24 записи гитары; **BPSD-dev** без Op. 53 — 44 записи фортепиано (сонаты Бетховена);
  **JazzSAMBA-val** — 12 записей джазовых ансамблей; **RWC-P** — 100 записей поп-музыки.

In [1]:
import importlib, inspect, json, os, sys, time
from pathlib import Path
import numpy as np
import torch
from IPython.display import HTML, display

ROOT = Path.cwd()
while not (ROOT / "chordrec").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
import chordrec

LEVELS = ["root", "majmin", "triads", "sevenths", "tetrads", "mirex"]
SEEDS, FOLDS = (42, 43, 44), range(5)


def load(path):
    return json.loads((ROOT / path).read_text(encoding="utf-8"))


def f2(v, sign=False, nd=2):
    s = f"{v:+.{nd}f}" if sign else f"{v:.{nd}f}"
    return s.replace(".", ",").replace("-", "−")


def cube(get):
    """[5 фолдов, 3 seeds, 6 уровней]"""
    return np.array([[[get(k, s)[q] for q in LEVELS] for s in SEEDS] for k in FOLDS])


def paired(a, b):
    """b − a: среднее 15 пар и число фолдов из 5 с плюсом."""
    d = (b - a).mean(1)
    return d.mean(0), (d > 0).sum(0)


def table(head, rows, note=None):
    th = "".join(f"<th>{h}</th>" for h in head)
    body = "".join("<tr>" + "".join(f"<td>{c}</td>" for c in r) + "</tr>" for r in rows)
    out = f"<table><thead><tr>{th}</tr></thead><tbody>{body}</tbody></table>"
    if note:
        out += f"<p style='font-size:90%;opacity:.75'>{note}</p>"
    display(HTML(out))


def where(obj):
    """Файл и строка начала функции или класса в пакете chordrec."""
    mod = inspect.getmodule(obj)
    return f"<code>chordrec/{mod.__name__.split('.')[-1]}.py:{inspect.getsourcelines(obj)[1]} {obj.__name__}</code>"


def line_of(module, marker):
    """Строка модуля, где стоит поясняющий комментарий marker."""
    for i, line in enumerate(inspect.getsource(module).splitlines(), 1):
        if marker in line:
            return f"<code>chordrec/{module.__name__.split('.')[-1]}.py:{i}</code>"
    raise KeyError(marker)


print("Пакет chordrec найден, PyTorch", torch.__version__)

Пакет chordrec найден, PyTorch 2.9.1+cu128


## 1. Структура проекта
| Папка | Что там |
|---|---|
| `chordrec/` | весь код итогового рецепта (таблица ниже строится из самих файлов) |
| `weights/` | 15 моделей (`fold{k}/s{seed}.pt`), нормировка входа каждого фолда, штраф декодера каждой модели (`models.json`) |
| `results/` | сохранённые числа исследовательского проекта, на которых строятся таблицы; источник каждого файла — `results/SOURCES.json` |
| `notebooks/` | этот ноутбук и его сборщик |
| `tests/` | проверка против исследовательского кода |
| `tools/` | упаковка весов и чисел из исследовательского проекта |

In [2]:
MODULES = ["config", "features", "model", "decode", "rules", "recognize", "labels", "losses", "data", "train", "cv", "evaluate"]
rows = []
for name in MODULES:
    mod = importlib.import_module(f"chordrec.{name}")
    src = inspect.getsource(mod).splitlines()
    rows.append([f"<code>chordrec/{name}.py</code>", len(src), (mod.__doc__ or "").strip().splitlines()[0]])
rows.append(["<b>всего</b>", sum(r[1] for r in rows), ""])
table(["Файл", "строк", "Что делает"], rows)

Файл,строк,Что делает
chordrec/config.py,57,Константы итогового рецепта и пути к данным.
chordrec/features.py,80,"Признаки: CQT-спектрограмма, окно бинов для сети, нормировка и правило громкости (iii)."
chordrec/model.py,104,Сеть итогового рецепта: свёрточная часть → 4 × (Transformer с RoPE + свёрточный модуль) → пять выходов.
chordrec/decode.py,99,Декодеры: Витерби со штрафом смены аккорда; выход «мажор/минор»; «набор высот» по 180 шаблонам (12 корней × 15 качеств).
chordrec/rules.py,187,Пять правил распознавания поверх обученной модели (без переобучения).
chordrec/recognize.py,54,Распознать аккорды в аудиофайле итоговой моделью со всеми пятью правилами.
chordrec/labels.py,156,"Цели обучения по кадрам: «мажор/минор» и корень, «набор высот» и бас, ноты; их сдвиг вместе с окном CQT."
chordrec/losses.py,92,"Потери итогового рецепта: сглаживание по консонансу, согласие транспозиций, дистилляция ансамбля."
chordrec/data.py,111,"Данные обучения итогового рецепта: GuitarSet (разбиение по фолду), синтетика AAM, три пула реального звука и ноты MAESTRO."
chordrec/train.py,218,"Обучение одной модели итогового рецепта (один фолд, один seed)."


## 2. Как устроено распознавание
1. **Признаки.** CQT: 22 050 Гц, шаг 512 (23 мс), 36 бинов на октаву, 288 бинов от F#0, log(1 + 100·|X|); сеть читает 252 бина,
   запас ±6 полутонов нужен для транспозиций.
2. **Сеть.** Свёрточная часть, затем 4 блока «Transformer с вращательными позициями (RoPE) + свёрточный модуль», ширина 192.
3. **Пять выходов по кадрам:** мажор/минор (25), «набор высот» (12 сигмоид: звучит ли класс высоты как звук аккорда), бас (13),
   корень (13), ноты (88 клавиш; только при обучении).
4. **Декодер.** Витерби по 180 шаблонам (12 корней × 15 качеств) и «нет аккорда»; обращение — если бас звук аккорда, но не корень.
5. **Пять правил распознавания** поверх модели (раздел 5).

In [3]:
from chordrec import decode, features, model as M, rules
d = load("results/models/fold0_s42.json")
table(["Часть", "Где в коде"], [
    ["Признаки CQT", where(features.cqt_from_signal)],
    ["Окно бинов и нормировка входа", where(features.crop_bins) + " · " + where(features.norm_stats)],
    ["Сеть целиком", where(M.ChordNet)],
    ["Блок Transformer с RoPE", where(M.Block)],
    ["Свёрточный модуль", where(M.ConvModule)],
    ["Витерби", where(decode.viterbi)],
    ["Декодер «мажор/минор»", where(decode.decode_flat)],
    ["Оценка 180 шаблонов", where(decode.pcs_emissions)],
    ["Декодер полных меток", where(decode.decode_pcs)],
], note=f"Параметров в одной модели: {d['params']:,}".replace(",", " "))

Часть,Где в коде
Признаки CQT,chordrec/features.py:10 cqt_from_signal
Окно бинов и нормировка входа,chordrec/features.py:39 crop_bins · chordrec/features.py:45 norm_stats
Сеть целиком,chordrec/model.py:70 ChordNet
Блок Transformer с RoPE,chordrec/model.py:42 Block
Свёрточный модуль,chordrec/model.py:24 ConvModule
Витерби,chordrec/decode.py:25 viterbi
Декодер «мажор/минор»,chordrec/decode.py:55 decode_flat
Оценка 180 шаблонов,chordrec/decode.py:62 pcs_emissions
Декодер полных меток,chordrec/decode.py:81 decode_pcs


## 3. Механизмы обучения: где в коде и что дали
Сначала — основа рецепта (ранние шаги; числа из отчётов того времени, тогда распознавание шло записью целиком). Затем — шаги
итогового рецепта, пересчитанные здесь из сохранённых файлов: 15 моделей шага против 15 моделей предыдущего, BPSD-dev и
JazzSAMBA-val окнами 10 с, без правил распознавания.

Каждый шаг принимался по правилу, записанному до запуска: выигрыш — BPSD sevenths или majmin ≥ +1,0 (4 из 5 фолдов), устойчивая
прибавка септим BPSD ≥ +0,3 (4/5) или гитара ≥ +0,3; без вреда — majmin и sevenths нигде не падают больше чем на 0,5.
Решения — только по валидации; тестовые наборы для решений не используются.

In [4]:
from chordrec import data, labels, losses, train as T
early = [
    ("Транспозиционная аугментация: окно CQT сдвигается на −5…+6 полутонов, метки — так же",
     line_of(T, "# транспозиционная аугментация") + " · " + where(labels.shift_targets) + " · " + where(labels.shift_pcs), "GuitarSet majmin +12,9 — главный механизм"),
    ("Синтетика AAM: 600 синтезированных треков, доля 0,10", where(data.aam) + " · " + where(data.aam_features), "MusicNet majmin +11 на негитарных инструментах"),
    ("Пул MusicNet: классика, аккорды и септимы из нот, доля 0,15", where(data.pool), "BPSD majmin 43,1 → 56,3 (+13,3, 5/5)"),
    ("Пул рояля Disklavier (MAPS, SMD, KRAISLER), доля 0,10", where(data.pool), "BPSD majmin +5,25 (5/5)"),
    ("Выход «набор высот» и декодер по 180 шаблонам", where(labels.pcs_targets) + " · " + where(decode.pcs_emissions), "BPSD sevenths +4,93 (5/5)"),
    ("Джазовые записи JazzSAMBA, доля 0,15", where(data.pool), "джаз sevenths +11; BPSD majmin +1,0"),
    ("Уточнённые метки рояля", "манифест данных (подготовка — в исследовательском проекте)", "BPSD около +2 по всем уровням (sevenths +2,0, 5/5)"),
    ("Голова корня рядом с «набором высот» (вес 2 в декодере)", where(decode.pcs_emissions), "гитара root +2,49 (5/5)"),
]
table(["Механизм", "Где в коде", "Что дал (по отчёту)"], [list(r) for r in early])

STEPS = [
    ("Исходный рецепт", "00_base", ""),
    ("Вращательные позиции (RoPE)", "01_rope", where(M.Block)),
    ("Последняя эпоха вместо выбора", "02_last_epoch", line_of(T, "# веса последней эпохи")),
    ("Голова нот (88 клавиш)", "03_note_head", where(labels.note_targets) + " · " + line_of(T, "# ноты сдвигаются с окном")),
    ("Самообучение, круг 1", "04_self_training", "манифест MusicNet с подписями учителя"),
    ("Круг 2 + согласие транспозиций", "05_round2_consistency", where(losses.consistency_loss) + " · " + line_of(T, "# вторая копия отрезка")),
    ("Сглаживание по консонансу", "06_consonance", where(losses.consonance_ce)),
    ("Дистилляция ансамбля", "07_distillation", where(losses.teacher_targets) + " · " + where(losses.distillation_loss)),
    ("Бас из нот", "08_bass_from_notes", where(labels.bass_from_notes)),
    ("Ноты MAESTRO в голове нот", "09_maestro_notes", where(data.maestro_notes)),
]
LAST = STEPS[-1][0]  # итоговый (замороженный) рецепт
win = {}
for name, f, _ in STEPS:
    w = load(f"results/path/{f}.json")
    win[name] = {ds: cube(lambda k, s: w[f"fold{k}|s{s}|{ds}|win10s"]) for ds in ("bpsd", "jazz")}
Mi, Si = LEVELS.index("majmin"), LEVELS.index("sevenths")
rows = []
for i, (name, _, code_ref) in enumerate(STEPS):
    b, j = win[name]["bpsd"], win[name]["jazz"]
    if i == 0:
        rows.append([name, "", f2(b[..., Mi].mean()), f2(b[..., Si].mean()), f2(j[..., Si].mean()), f2(b[..., Si].mean())])
        continue
    pb, nb = paired(win[STEPS[i - 1][0]]["bpsd"], b)
    pj, _ = paired(win[STEPS[i - 1][0]]["jazz"], j)
    rows.append([name, code_ref, f"{f2(pb[Mi], True)} ({nb[Mi]})", f"{f2(pb[Si], True)} ({nb[Si]})", f2(pj[Si], True), f2(b[..., Si].mean())])
table(["Шаг", "Где в коде", "BPSD majmin", "BPSD sevenths", "Джаз sevenths", "BPSD sevenths, итого"], rows,
      note="Разница с предыдущей строкой, в скобках — фолдов > 0 из 5; первая строка — абсолютные значения. "
           "Круг 2 + согласие — ниже правила, принят решением автора (все шесть уровней BPSD выросли в 5/5); консонанс и дистилляция — "
           "по дополнению о септимах; бас из нот — по гитаре (balanced +0,59, 5/5); ноты MAESTRO (с дистилляцией) — по правилу.")

Механизм,Где в коде,Что дал (по отчёту)
"Транспозиционная аугментация: окно CQT сдвигается на −5…+6 полутонов, метки — так же",chordrec/train.py:110 · chordrec/labels.py:53 shift_targets · chordrec/labels.py:90 shift_pcs,"GuitarSet majmin +12,9 — главный механизм"
"Синтетика AAM: 600 синтезированных треков, доля 0,10",chordrec/data.py:66 aam · chordrec/data.py:25 aam_features,MusicNet majmin +11 на негитарных инструментах
"Пул MusicNet: классика, аккорды и септимы из нот, доля 0,15",chordrec/data.py:77 pool,"BPSD majmin 43,1 → 56,3 (+13,3, 5/5)"
"Пул рояля Disklavier (MAPS, SMD, KRAISLER), доля 0,10",chordrec/data.py:77 pool,"BPSD majmin +5,25 (5/5)"
Выход «набор высот» и декодер по 180 шаблонам,chordrec/labels.py:65 pcs_targets · chordrec/decode.py:62 pcs_emissions,"BPSD sevenths +4,93 (5/5)"
"Джазовые записи JazzSAMBA, доля 0,15",chordrec/data.py:77 pool,"джаз sevenths +11; BPSD majmin +1,0"
Уточнённые метки рояля,манифест данных (подготовка — в исследовательском проекте),"BPSD около +2 по всем уровням (sevenths +2,0, 5/5)"
Голова корня рядом с «набором высот» (вес 2 в декодере),chordrec/decode.py:62 pcs_emissions,"гитара root +2,49 (5/5)"


Шаг,Где в коде,BPSD majmin,BPSD sevenths,Джаз sevenths,"BPSD sevenths, итого"
Исходный рецепт,,"70,01","54,39","37,86","54,39"
Вращательные позиции (RoPE),chordrec/model.py:42 Block,"+0,42 (3)","+1,03 (4)","+0,46","55,42"
Последняя эпоха вместо выбора,chordrec/train.py:186,"+0,90 (5)","+0,84 (4)","+2,00","56,26"
Голова нот (88 клавиш),chordrec/labels.py:111 note_targets · chordrec/train.py:120,"+0,46 (5)","+0,68 (4)","+0,47","56,94"
"Самообучение, круг 1",манифест MusicNet с подписями учителя,"+0,50 (5)","+1,38 (5)","+0,42","58,32"
Круг 2 + согласие транспозиций,chordrec/losses.py:34 consistency_loss · chordrec/train.py:138,"+0,90 (5)","+0,71 (5)","−0,63","59,03"
Сглаживание по консонансу,chordrec/losses.py:12 consonance_ce,"+0,07 (3)","+0,31 (5)","+0,85","59,34"
Дистилляция ансамбля,chordrec/losses.py:64 teacher_targets · chordrec/losses.py:86 distillation_loss,"+0,37 (5)","+0,53 (5)","+0,70","59,87"
Бас из нот,chordrec/labels.py:134 bass_from_notes,"+0,09 (5)","+0,24 (5)","+0,52","60,11"
Ноты MAESTRO в голове нот,chordrec/data.py:93 maestro_notes,"+0,12 (5)","+0,68 (5)","+0,43","60,78"


## 4. Обучение
Одна модель — `python -m chordrec.train --fold 0 --seed 42 --out runs/final`; все 15 — `python -m chordrec.cv --out runs/final`.
Рецепт: 60 эпох × 100 шагов, батч 16 отрезков по 10 с, AdamW (lr 0,001, weight decay 0,01), OneCycle, веса последней эпохи.
Доли в батче: GuitarSet 0,45 · синтетика AAM 0,10 · MusicNet 0,15 · рояль 0,10 · JazzSAMBA 0,15 · ноты MAESTRO 0,05 (только голова нот).
Обучение одной модели — около 35 минут на RTX 4060 Laptop плюс модели-учителя; данные (десятки гигабайт) читаются из
исследовательского проекта, часть наборов по лицензии распространять нельзя.

In [5]:
table(["Слагаемое потери", "Вес", "Где в коде"], [
    ["«Мажор/минор»: CE", "1", line_of(T, 'ce(o["flat"]')],
    ["Корень и бас: CE со сглаживанием по консонансу, α = 0,1", "1 + 1", where(losses.consonance_ce)],
    ["«Набор высот»: BCE по 12 классам", "12", line_of(T, 'o["pc"].float()[m]')],
    ["Ноты: BCE по 88 клавишам", "12", line_of(T, 'o["note"].float()[m]')],
    ["Дистилляция: 3 модели учителя × 3 сдвига", "1", where(losses.distillation_loss)],
    ["Согласие двух транспозиций отрезка", "0,5", where(losses.consistency_loss)],
])

Слагаемое потери,Вес,Где в коде
«Мажор/минор»: CE,1,chordrec/train.py:149
"Корень и бас: CE со сглаживанием по консонансу, α = 0,1",1 + 1,chordrec/losses.py:12 consonance_ce
«Набор высот»: BCE по 12 классам,12,chordrec/train.py:154
Ноты: BCE по 88 клавишам,12,chordrec/train.py:161
Дистилляция: 3 модели учителя × 3 сдвига,1,chordrec/losses.py:86 distillation_loss
Согласие двух транспозиций отрезка,"0,5",chordrec/losses.py:34 consistency_loss


## 5. Пять правил распознавания (без переобучения)

In [6]:
table(["Правило", "Что делает", "Где в коде"], [
    ["(i) K = 3", "вход сдвигается на −1, 0, +1 полутон, ответы возвращаются к исходной тональности и усредняются", where(rules.unshift) + " · " + where(rules.recognize)],
    ["(ii) штраф 96", "Витерби реже меняет аккорд на коротких всплесках", "<code>rules.PENALTY</code> · " + where(decode.viterbi)],
    ["(iii) громкость", "окно 10 с громче или тише [−12,2; +8,5] дБ подтягивается к границе", where(features.clamp_level)],
    ["(iv) тональность → функция", "первый проход → тональность по своим аккордам → второй проход с прибавкой β·W", where(rules.frame_keys) + " · " + where(rules.decode_with_key)],
    ["(v) перекрывающиеся окна", "окна 10 с с шагом 2 с; кадру — окно, где он ближе к центру", where(rules.center_probs)],
])
ladder, summ = load("results/rules/ladder.json"), load("results/rules/final_rows_dev_summary.json")
SETS = [("bpsd", "BPSD"), ("rwcp", "RWC-P"), ("jazz", "Джаз"), ("guitar", "Гитара")]
LADDER = [("Одна модель, окна 10 с", "single"), ("+ (i) K = 3", "i_K3"), ("+ (ii) штраф 96", "ii_pen96"),
          ("+ (iii) громкость", "iii_clamp"), ("+ (iv) тональность → функция", "iv_key"), ("+ (v) перекрывающиеся окна", "v_overlap")]
rows = []
for i, (n, key) in enumerate(LADDER):
    cells = [n]
    for ds, _ in SETS:
        v = ladder[f"{ds}|{key}"]["sevenths"]
        cells.append(f2(v) if i == 0 else f"{f2(v)} ({f2(ladder[f'{ds}|{key}|delta']['sevenths'], True)})")
    rows.append(cells)
table(["Распознавание", "BPSD", "RWC-P", "Джаз", "Гитара"], rows,
      note="sevenths, среднее 15 моделей итогового рецепта; правила добавляются по одному; в скобках — прибавка к предыдущей строке. "
           "Гитара — отложенные группы кросс-валидации. Цена: правило (i) — распознавание × 3, правило (v) — ещё × 5.")

Правило,Что делает,Где в коде
(i) K = 3,"вход сдвигается на −1, 0, +1 полутон, ответы возвращаются к исходной тональности и усредняются",chordrec/rules.py:56 unshift · chordrec/rules.py:169 recognize
(ii) штраф 96,Витерби реже меняет аккорд на коротких всплесках,rules.PENALTY · chordrec/decode.py:25 viterbi
(iii) громкость,"окно 10 с громче или тише [−12,2; +8,5] дБ подтягивается к границе",chordrec/features.py:71 clamp_level
(iv) тональность → функция,первый проход → тональность по своим аккордам → второй проход с прибавкой β·W,chordrec/rules.py:151 frame_keys · chordrec/rules.py:159 decode_with_key
(v) перекрывающиеся окна,"окна 10 с с шагом 2 с; кадру — окно, где он ближе к центру",chordrec/rules.py:38 center_probs


Распознавание,BPSD,RWC-P,Джаз,Гитара
"Одна модель, окна 10 с","60,78","57,20","43,07","87,34"
+ (i) K = 3,"61,34 (+0,55)","58,05 (+0,85)","43,42 (+0,35)","88,01 (+0,67)"
+ (ii) штраф 96,"61,56 (+0,22)","58,40 (+0,36)","42,94 (−0,48)","89,28 (+1,27)"
+ (iii) громкость,"61,62 (+0,07)","60,71 (+2,31)","42,95 (+0,01)","89,30 (+0,01)"
+ (iv) тональность → функция,"62,39 (+0,77)","61,92 (+1,21)","43,72 (+0,77)","88,91 (−0,38)"
+ (v) перекрывающиеся окна,"62,65 (+0,25)","61,75 (−0,18)","44,21 (+0,49)","89,61 (+0,70)"


## 6. Итог и сравнение с открытыми системами
Открытые системы — опубликованные веса (MIT) и их собственный код, без дообучения. Все три обучались на RWC-Pop, поэтому
на RWC-P не сравниваются. Обучающие части GuitarSet и JazzSAMBA входят в наши данные: DEV и джаз для нас свой домен,
для открытых систем — нет. BPSD-final и JazzSAMBA-test открыты один раз после заморозки рецепта (раздел 10);
95 % бутстреп-интервалы разниц — в `results/final/report_*.json` и в статье.

In [7]:
OPEN = [("Chord-CNN-LSTM (2019)", "chord_cnn_lstm"), ("BTC (2019)", "btc"), ("ChordMini (2026)", "chordmini")]
openv = {key: load(f"results/open_systems/{key}.json") for _, key in OPEN}


def dev_ours():
    vals = {q: [] for q in LEVELS}
    for k in FOLDS:
        for s in SEEDS:
            lv = load(f"results/models/fold{k}_s{s}.json")["levels"]
            for q in LEVELS:
                src = lv[f"CV-Rmnds-s{s}/flat/viterbi" if q == "majmin" else f"CV-Rmnds-s{s}/struct/viterbi"]["comp"][q]
                vals[q].append(src["wcsr"] if isinstance(src, dict) else src)
    return [float(np.mean(vals[q])) for q in LEVELS]


def five_rules(ds):
    return [summ[f"{ds}|rules"]["mean"][q] for q in LEVELS]


rows = []
for title, key, ours, ours_r in [("GuitarSet DEV (24)", "dev", dev_ours(), None),
                                 ("BPSD-dev без Op. 53 (44)", "bpsd", list(win[LAST]["bpsd"].mean((0, 1))), five_rules("bpsd")),
                                 ("JazzSAMBA-val (12)", "jazz", list(win[LAST]["jazz"].mean((0, 1))), five_rules("jazz"))]:
    rows.append([f"<b>{title}</b>"] + [""] * 6)
    cand = [[openv[k][key]["levels"][q] for q in LEVELS] for _, k in OPEN] + [ours] + ([ours_r] if ours_r else [])
    names = [n for n, _ in OPEN] + ["наша, одна модель"] + (["наша + 5 правил"] if ours_r else [])
    best = np.max(np.array(cand), 0)
    for n, v in zip(names, cand):
        rows.append([n] + [f"<b>{f2(x)}</b>" if abs(x - b) < 1e-9 else f2(x) for x, b in zip(v, best)])
rows.append(["<b>Только наша система</b>"] + [""] * 6)
rows.append(["RWC-P (100), наша + 5 правил"] + [f2(x) for x in five_rules("rwcp")])
rows.append(["GuitarSet, отложенные группы, наша + 5 правил"] + [f2(x) for x in five_rules("guitar")])
table(["Набор и система"] + LEVELS, rows, note="Жирным — лучшее в столбце внутри набора. Наша система — среднее 15 моделей.")

Набор и система,root,majmin,triads,sevenths,tetrads,mirex
GuitarSet DEV (24),,,,,,
Chord-CNN-LSTM (2019),"90,33","91,68","58,40","84,29","51,38","87,38"
BTC (2019),"83,77","87,21","54,49","78,45","46,39","80,14"
ChordMini (2026),"84,86","87,68","54,69","80,94","48,04","82,58"
"наша, одна модель","93,11","93,73","61,55","90,07","57,40","90,81"
BPSD-dev без Op. 53 (44),,,,,,
Chord-CNN-LSTM (2019),"53,27","58,66","51,53","47,30","41,02","55,26"
BTC (2019),"63,01","68,10","60,59","59,29","52,69","68,19"
ChordMini (2026),"62,97","67,76","60,49","58,90","52,29","68,53"
"наша, одна модель","66,87","73,85","63,45","60,78","53,47","71,04"


## 7. Согласованность по тональностям (главный вывод статьи)
Звук 11 записей BPSD-dev (по одной на сонату) сдвигается на −4, −1, +1, +4 полутона; ответ сдвигается обратно и сравнивается
с ответом без сдвига. Наша система — итоговый рецепт без правил, пять моделей (по одной на фолд).

In [8]:
shift, vote, singles = load("results/consistency/audio_shift.json"), load("results/consistency/vote.json"), load("results/consistency/singles.json")


def dis(key):
    a = shift[key]["audio"]
    return [float(np.mean([a[s][i] for s in a])) for i in (0, 1)]


rows = []
for name, key in [("BTC", "btc_teacher"), ("ChordMini, ученик BTC", "btc_cl"), ("ChordMini, ChordNet", "chordnet")]:
    c, e = dis(key)
    rows.append([name, 1, f2(c, nd=1), f2(e, nd=1), f2(vote[key]["K1"]["sevenths"]), f2(vote[key]["K5"]["sevenths"]),
                 f2(vote[key]["K5"]["sevenths"] - vote[key]["K1"]["sevenths"], True)])
c, e = singles["cnnlstm_single_mean"]
rows.append(["Chord-CNN-LSTM, одна модель", 1, f2(c, nd=1), f2(e, nd=1), f2(singles["levels_s0"]["cnnlstm_single_mean"]["sevenths"]), "—", "—"])
c, e = dis("cnnlstm")
rows.append(["Chord-CNN-LSTM, ансамбль 5 моделей", 5, f2(c, nd=1), f2(e, nd=1), f2(vote["cnnlstm"]["K1"]["sevenths"]),
             f2(vote["cnnlstm"]["K5"]["sevenths"]), f2(vote["cnnlstm"]["K5"]["sevenths"] - vote["cnnlstm"]["K1"]["sevenths"], True)])
ours = load("results/consistency/ours_audio_shift.json")  # итоговый рецепт CV-Rmnds
c, e = ours["single_D"]
k1, k5 = ours["single_levels"]["K1"]["sevenths"], ours["single_levels"]["K5"]["sevenths"]
rows.append(["наша, одна модель", 1, f2(c, nd=1), f2(e, nd=1), f2(k1), f2(k5), f2(k5 - k1, True)])
c, e = ours["ensemble_D"]
rows.append(["наша, ансамбль 5 моделей", 5, f2(c, nd=1), f2(e, nd=1), f2(ours["ensemble_levels"]["K1"]["sevenths"]), "—", "—"])
table(["Система", "моделей", "меняет ответ, грубо, %", "точно, %", "sevenths", "голосование 5 сдвигов", "прирост"], rows,
      note="«Грубо» — основной тон и род аккорда, «точно» — полная метка; среднее по четырём сдвигам. Выборка мала (11 записей), числа ориентировочные.")

Система,моделей,"меняет ответ, грубо, %","точно, %",sevenths,голосование 5 сдвигов,прирост
BTC,1,"24,5","30,4","58,97","60,64","+1,67"
"ChordMini, ученик BTC",1,"23,8","32,0","57,58","60,16","+2,58"
"ChordMini, ChordNet",1,"24,5","30,6","56,63","58,20","+1,57"
"Chord-CNN-LSTM, одна модель",1,"27,6","43,4","47,21",—,—
"Chord-CNN-LSTM, ансамбль 5 моделей",5,"16,4","27,3","50,73","50,54","−0,19"
"наша, одна модель",1,"18,2","29,9","60,00","60,03","+0,03"
"наша, ансамбль 5 моделей",5,"16,6","27,7","60,36",—,—


## 8. Проверка против исследовательского кода
`tests/check_against_research.py` загружает те же веса в чистый и в исследовательский код. Нормировка входа совпадает побитно,
выходы сети — с расхождением 0, а распознавание с пятью правилами и без них даёт те же аккорды на записях всех четырёх
наборов. С ключом `--full` чистый код пересчитывает всё BPSD-dev для одной модели и сравнивает уровни с сохранёнными.
`tests/check_training.py` обучает модель 20 шагов исследовательским и чистым кодом с одним seed и сравнивает результат.

In [9]:
for name in ("last_check.txt", "last_training_check.txt"):
    p = ROOT / "tests" / name
    print(p.read_text(encoding="utf-8") if p.exists() else f"{name}: отчёт ещё не сохранён")

Проверка распознавания (tests/check_against_research.py --full, 20.11.2026): модель фолда 0, seed 42; «overlap» — все пять правил, «plain» — без правил.
1. нормировка фолда 0 совпадает побитно: True
2. модель: наибольшее расхождение выходов 0.0
3. bpsd bpsd:Beethoven_Op002No2-01_AS35_comp: правила 585 отрезков, без правил 923 — совпадают: True
3. bpsd bpsd:Beethoven_Op002No2-01_FG58_comp: правила 520 отрезков, без правил 886 — совпадают: True
3. jazz jazzsamba:sync:57-chega_de_saudade:alternate_comp: правила 692 отрезков, без правил 1160 — совпадают: True
3. jazz jazzsamba:sync:57-chega_de_saudade:preferred_comp: правила 680 отрезков, без правил 1113 — совпадают: True
3. rwcp rwc:P001_comp: правила 189 отрезков, без правил 277 — совпадают: True
3. rwcp rwc:P002_comp: правила 172 отрезков, без правил 256 — совпадают: True
3. guitar 00_Funk2-108-Eb_comp: правила 28 отрезков, без правил 32 — совпадают: True
3. guitar 00_Funk2-119-G_comp: правила 33 отрезков, без правил 37 — совпадают: Tru

## 9. Распознавание аккордов прямо здесь
Запись GuitarSet DEV (гитара, джаз, Bb, 35 с), одна модель (фолд 0, seed 42), все пять правил, процессор. Признаки считаются
из аудиофайла заново. Из командной строки то же самое: `python -m chordrec.recognize запись.wav`.

In [10]:
import librosa
from chordrec.config import MANIFESTS, SR
from chordrec.evaluate import levels as eval_levels
from chordrec.features import cqt_from_signal
from chordrec.recognize import load_models

rec = next(json.loads(l) for l in open(MANIFESTS["guitarset"], encoding="utf-8") if '"00_Jazz2-110-Bb_comp"' in l)
y, _ = librosa.load(rec["audio_path"], sr=SR, mono=True)
cpu = torch.device("cpu")
t0 = time.time()
x = cqt_from_signal(y)
res = rules.recognize(load_models(1, cpu), x, len(y) / SR, cpu, rules=True)
print(f"Запись {len(y) / SR:.1f} с, распознана за {time.time() - t0:.1f} с на процессоре")


def ref_at(t):
    return next((s["label"] for s in rec["chord_intervals"] if s["start"] <= t < s["end"]), "N")


import mir_eval

merged = []  # соседние отрезки одного аккорда (различаются только басом) склеиваются для показа
for s in res["chords"]:
    lab = s["label"].split("/")[0]
    if merged and merged[-1]["label"] == lab:
        merged[-1]["end"] = s["end"]
    else:
        merged.append({"start": s["start"], "end": s["end"], "label": lab})
rows = []
for s in merged[:14]:
    r = ref_at((s["start"] + s["end"]) / 2)
    sc = mir_eval.chord.sevenths([r], [s["label"]])[0]
    rows.append([f"{s['start']:.1f}–{s['end']:.1f}", r, s["label"], "✓" if sc == 1 else ("вне словаря" if sc < 0 else "✗")])
table(["Время, с", "Эталон (в середине отрезка)", "Распознано", "верно на уровне sevenths"], rows,
      note="Первые 14 аккордов. Обозначения Harte: C:maj7 — большой мажорный септаккорд, D:min7 — малый минорный, C:7 — доминантсептаккорд; "
           "/1, /5 — бас. «Вне словаря» — эталонный аккорд не входит в словарь уровня sevenths (например, sus2 с септимой) и в оценке не участвует.")
lv = eval_levels([rec], {rec["id"]: res["majmin"]}, {rec["id"]: res["chords"]})
table(LEVELS, [[f2(lv[q]) for q in LEVELS]], note="Уровни на этой записи (одна модель, пять правил).")

Запись 34.9 с, распознана за 4.9 с на процессоре


"Время, с",Эталон (в середине отрезка),Распознано,верно на уровне sevenths
0.0–2.5,D#:sus2(b7)/1,D#:min7,вне словаря
2.5–4.4,G#:min7/1,G#:7,✗
4.4–6.4,C#:maj7/5,C#:7,✗
6.4–9.0,F#:maj/5,F#:maj,✓
9.0–10.8,C:hdim7/1,C:hdim7,вне словаря
10.8–13.1,"F:7(#9,*5)/1",F:7,вне словаря
13.1–17.4,A#:min7/1,A#:min7,✓
17.4–20.2,D#:sus4(b7)/1,D#:min7,вне словаря
20.2–21.6,G#:7/1,G#:7,✓
21.6–24.0,C#:7/1,C#:7,✓


root,majmin,triads,sevenths,tetrads,mirex
"94,79","85,38","58,47","76,20","52,81","94,79"


## 10. Отложенные наборы
**BPSD-final** (80 записей, 20 сонат) и **JazzSAMBA-test** (14 записей, 7 пьес) открыты один раз после заморозки рецепта:
протокол с SHA-256 весов и скриптов — `results/final/protocol_frozen.json`. Те же 15 моделей и те же три открытые системы;
95 % интервалы — бутстреп по сонатам (пьесам), 10 000 повторов. GuitarSet TEST не используется.

In [11]:
fin = load("results/final/report_final.json")
NAMES = [("cnnlstm", "Chord-CNN-LSTM"), ("btc_teacher", "BTC"), ("btc_cl", "ChordMini"), ("ours_single", "наша, одна модель"),
         ("ours_rules", "наша + 5 правил")]
rows = []
for ds, title in [("bpsd_final", "BPSD-final (80)"), ("jazz_test", "JazzSAMBA-test (14)")]:
    rows.append([f"<b>{title}</b>"] + [""] * 6)
    lv = fin[f"{ds}|levels"]
    best = np.max(np.array([[lv[k][q] for q in LEVELS] for k, _ in NAMES]), 0)
    for k, n in NAMES:
        rows.append([n] + [f"<b>{f2(lv[k][q])}</b>" if abs(lv[k][q] - b) < 1e-9 else f2(lv[k][q]) for q, b in zip(LEVELS, best)])
table(["Набор и система"] + LEVELS, rows, note="Жирным — лучшее в столбце внутри набора.")
rows = []
for ds, title in [("bpsd_final", "BPSD-final"), ("jazz_test", "JazzSAMBA-test")]:
    for k, n in [("btc_teacher", "BTC"), ("cnnlstm", "Chord-CNN-LSTM")]:
        d = fin[f"{ds}|ours_rules-{k}"]
        rows.append([title, n] + [f"{f2(d[q]['diff'], True)} [{f2(d[q]['ci95'][0])}; {f2(d[q]['ci95'][1])}]" for q in ("majmin", "sevenths", "tetrads")])
table(["Набор", "против", "majmin", "sevenths", "tetrads"], rows, note="Наша + 5 правил минус открытая система, 95 % интервал.")

Набор и система,root,majmin,triads,sevenths,tetrads,mirex
BPSD-final (80),,,,,,
Chord-CNN-LSTM,"56,07","59,61","54,70","47,92","43,61","58,17"
BTC,"64,28","68,13","62,59","58,12","53,36","68,19"
ChordMini,"63,56","67,40","61,80","57,61","52,76","68,34"
"наша, одна модель","70,24","75,06","67,60","60,49","55,69","72,89"
наша + 5 правил,"70,71","75,74","69,10","61,70","56,83","73,31"
JazzSAMBA-test (14),,,,,,
Chord-CNN-LSTM,"63,98","57,63","55,29","37,66","34,74","64,52"
BTC,"56,70","49,24","47,04","21,30","19,54","53,68"
ChordMini,"59,46","51,84","49,59","27,48","25,11","58,31"


Набор,против,majmin,sevenths,tetrads
BPSD-final,BTC,"+7,61 [5,63; 9,62]","+3,58 [1,18; 5,95]","+3,46 [1,31; 5,61]"
BPSD-final,Chord-CNN-LSTM,"+16,13 [12,71; 19,62]","+13,79 [10,82; 16,81]","+13,22 [10,48; 16,06]"
JazzSAMBA-test,BTC,"+13,62 [10,00; 18,93]","+20,00 [10,19; 28,99]","+19,04 [10,30; 27,81]"
JazzSAMBA-test,Chord-CNN-LSTM,"+5,23 [2,29; 7,78]","+3,64 [−4,47; 10,70]","+3,84 [−3,85; 9,85]"
